# PR1: Intelligent Agents and PEAS (Smart Home Controller)

**Aim:** Build two intelligent agents for the same task, show the PEAS description and compare them.

| Agent | Idea |
|---|---|
| **Simple reflex agent** | Looks at the current percept only (no memory). |
| **Model-based agent** | Keeps an internal state (memory) of the world and uses it with the percept. |

## PEAS table

| Letter | Meaning | Smart Home Controller |
|---|---|---|
| **P** | Performance measure | Comfort of the room, energy saved |
| **E** | Environment | The house: room temperature, light, people |
| **A** | Actuators | AC, Heater, Light switch, Fan |
| **S** | Sensors | Temperature sensor, Light sensor, Motion sensor |

## Agent function

Percept = (temperature, is_dark, someone_in_room). The agent returns a list of actions.

In [ ]:
def smart_home_agent(temp, is_dark, person_in_room):
    actions = []

    # Rules for temperature
    if temp > 30:
        actions.append("Turn ON AC")
    elif temp < 18:
        actions.append("Turn ON Heater")
    else:
        actions.append("Temperature OK")

    # Rules for light
    if person_in_room and is_dark:
        actions.append("Turn ON Light")
    elif not person_in_room:
        actions.append("Turn OFF Light")

    return actions

## Environment simulation

Each tuple is one percept: `(temperature, is_dark, person_in_room)`.

In [ ]:
percepts = [
    (35, True,  True),
    (15, True,  True),
    (24, False, True),
    (28, True,  False),
    (32, False, False),
]

for p in percepts:
    result = smart_home_agent(p[0], p[1], p[2])
    print("Percept:", p, "->", result)

## Performance measure

Score +1 for every correct action, using the expected answers below.

In [ ]:
expected_first_action = ["Turn ON AC", "Turn ON Heater", "Temperature OK",
                         "Temperature OK", "Turn ON AC"]

score = 0
for i in range(len(percepts)):
    p = percepts[i]
    action = smart_home_agent(p[0], p[1], p[2])[0]
    if action == expected_first_action[i]:
        score = score + 1

print("Score:", score, "out of", len(percepts))

---
# Agent 2: Model-Based Reflex Agent

It remembers things in a `state` dictionary:

| Remembered | Used for |
|---|---|
| `ac_on`, `heater_on`, `light_on` | Do not repeat an action that is already done. |
| `last_temp` | If the temperature sensor fails (`None`), use the last known value. |

```
Percept + State  -->  rules  -->  Action  -->  update State
```

In [ ]:
state = {"ac_on": False, "heater_on": False, "light_on": False, "last_temp": 25}

def model_based_agent(temp, is_dark, person_in_room):
    actions = []

    # 1. sensor failed? use the remembered value
    if temp is None:
        temp = state["last_temp"]
    state["last_temp"] = temp

    # 2. temperature rules: act only if the state has to change
    if temp > 30 and not state["ac_on"]:
        actions.append("Turn ON AC")
        state["ac_on"] = True
        state["heater_on"] = False
    elif temp < 18 and not state["heater_on"]:
        actions.append("Turn ON Heater")
        state["heater_on"] = True
        state["ac_on"] = False
    elif 18 <= temp <= 30 and (state["ac_on"] or state["heater_on"]):
        actions.append("Turn OFF AC/Heater")
        state["ac_on"] = False
        state["heater_on"] = False

    # 3. light rules
    if person_in_room and is_dark and not state["light_on"]:
        actions.append("Turn ON Light")
        state["light_on"] = True
    elif not person_in_room and state["light_on"]:
        actions.append("Turn OFF Light")
        state["light_on"] = False

    if len(actions) == 0:
        actions.append("No change")

    return actions

## Compare both agents on the same percept sequence

`None` means the temperature sensor failed.

In [ ]:
sequence = [
    (35, True,  True),     # hot, dark, person in room
    (36, True,  True),     # still hot: nothing new to do
    (None, True, True),    # temperature sensor fails
    (24, False, False),    # comfortable, room empty
    (15, True,  True),     # cold again
]

for p in sequence:
    if p[0] is None:
        reflex = "cannot decide (no memory)"
    else:
        reflex = smart_home_agent(p[0], p[1], p[2])
    model = model_based_agent(p[0], p[1], p[2])

    print("Percept    :", p)
    print("Reflex     :", reflex)
    print("Model-based:", model)
    print()

## Reflex vs Model-based

| | Simple reflex | Model-based |
|---|---|---|
| Uses | Current percept only | Percept + internal state |
| Memory | No | Yes (`state`) |
| Repeats actions? | Yes (turns AC on again at 36) | No (`No change`) |
| Sensor failure | Cannot cope | Uses `last_temp` |
| Complexity | Very simple | A bit more |

## Viva points

- Agent = anything that **perceives** the environment through sensors and **acts** through actuators.
- **Simple reflex agent:** uses only the **current percept** (no memory). Rules are if-else.
- **Model-based agent:** keeps an **internal state** (a model of the world) and updates it after every percept.
- Use model-based when the environment is **partially observable** (the agent cannot see everything at once).
- PEAS describes the task environment of an agent.
- This environment is **partially observable, dynamic, deterministic** (the rules always give the same output).
- Other agent types: goal-based, utility-based, learning agent.